# Milestone - Unit 11: robustness

Unit 11 asked what Unit 10's agent does on cars that are not the one it was tuned on, and how to survive a crash.
`agents/unit-11.py` is Unit 10's file (the racing line, the measured grip, the speed profile, LTV least-squares MPC with the measured lag of 1 tick) with:

- `MU_FRACTION = 0.54`, re-tuned on the perturbed training family (cars 0-23 of `carla_like(perturb=k)`) by the margin rule and the expected race time;
- a respawn handler, edge-triggered on the collision sensor's impulse (over `RESPAWN_IMPULSE` = 100 N s, not the previous step's reading, none handled in the last `REFRACTORY_TICKS` = 40 steps);
- `fall_back()`: clear the MPC's lag queue, last command and plan, re-anchor the waypoint index on the next step, switch to pure pursuit (`FALLBACK_STEERING`) and multiply the lateral limit by `FALLBACK_SCALE` = 0.86 at every respawn, re-planning the speed profile;
- the instruments: the sideslip and the line error, counted in `warnings`, never acted on (`WARN_FALLBACK = False`).

The notebook asserts the unit's gates (plan `roar-010`) on the small subset below; the routed racekit test (`tests/test_racekit_unit11_premise.py`) races every 24-car figure in full (218 races):

- **U11.6** the gate, held-out: the shipped file is clean on held-out cars 35 and 41 (the premise: on all 24 held-out cars, k = 24-47);
- **U11.5** the fallback: at `MU_FRACTION = 0.58` on training car 20, where Unit 10 crash-loops, it finishes in 385.05 s with exactly 1 collision and 1 respawn handled, in fallback mode;
- **U11.3** respawn economics: the race after that respawn (race time - crash time - 1 s) is within 1 s of a fresh race of car 20 in the fallback configuration (pure pursuit at 0.58 x 0.86: 285.35 s), and shorter (the speed loop's integral survives the restart);
- **U11.5** the lag-2 car (`carla_like(noise=True, lag_ticks=2)`: the lag Unit 8 identified, plus one tick): Unit 10 at 0.54 does not finish (a crash loop, at least 3 respawns in 450 s), and the shipped file finishes in 321.65 s with exactly 1 collision;
- **U11.7** equivalence: with no respawn the shipped file races exactly like `agents/unit-10.py` at 0.54 on the unperturbed car, row for row (279.65 s), with 0 respawns, 0 warnings, in primary mode;
- **U11.7a** the handler, race-free: a repeated impulse is handled once, a new one inside the 40-step window is not, one after it is, and `fall_back()` resets and degrades as stated; `check_submission` passes.

Every race uses racekit's carla-like car with its noise, `carla_like(noise=True, perturb=k)`, on `test-circuit`, race seed k mod 5 (seed 0 for an unperturbed car), with `max_seconds` 450.
Every MPC race runs the variant `MPC_BUDGET_MS = None` (no wall clock inside a race, as in Unit 10).
Tolerances (the plan's Gates): times within two ticks (±0.10 s) of the pinned value; outcomes and collision counts exact where the plan pins them; "does not finish" for a crash loop.
A race is **clean** when it finished, with 0 collisions, and its max |cross-track error from its own line| over the driving rows is at most 2 m.
Every figure is a model number on racekit's carla-like car, not CARLA's.

## 1. Set-up

The milestone runs from `projects/racekit/milestones/`, so the agents are one directory up.

In [ ]:
from pathlib import Path
from tempfile import mkdtemp

import numpy as np
from racekit import write_variant
from racekit.cars import carla_like
from racekit.check_submission import check_submission
from racekit.runner import load_submission, run_race, shim_installed
from racekit.tracks import load_track

AGENTS = Path("..") / "agents"
UNIT_09, UNIT_10, UNIT_11 = AGENTS / "unit-09.py", AGENTS / "unit-10.py", AGENTS / "unit-11.py"
TICKS_2 = 0.10 + 1e-9  # s: two ticks, the plan's tolerance
CLEAN_ERROR = 2.0  # m
HOLD_S = 1.0  # s: the respawn hold
CAP = 450.0  # s
NO_TIMING = {"MPC_BUDGET_MS": None}
workdir = Path(mkdtemp(prefix="roar-milestone-11-"))

with shim_installed():
    Unit11, _ = load_submission(UNIT_11)
track = load_track("test-circuit")
offsets = np.array(Unit11(track.waypoints, None).offsets)  # test-circuit's line as shipped


def variant(agent, name, **constants):
    return write_variant(agent, workdir / f"{name}.py", {key: repr(value) for key, value in constants.items()})


def line_error(result):
    telemetry = result.telemetry
    driving = telemetry["held"] == 0
    target = offsets[(telemetry["progress"][driving] + 1) % len(offsets)]
    return float(np.max(np.abs(telemetry["offset"][driving] - target)))


def clean(result):
    return result.finished and result.collisions == 0 and line_error(result) <= CLEAN_ERROR


def kept(source):
    # The submission's class, keeping its instance (to read its counters after the race).
    with shim_installed():
        Agent, _ = load_submission(source)
    agents = []

    class Kept(Agent):
        def __init__(self, *args, **kwargs):
            super().__init__(*args, **kwargs)
            agents.append(self)

    return Kept, agents


def race(source, k=None, lag_ticks=None, seed=None):
    if seed is None:
        seed = 0 if k is None else k % 5
    return run_race(source, "test-circuit", car=carla_like(noise=True, perturb=k, lag_ticks=lag_ticks), seed=seed,
                    max_seconds=CAP)


unit10_054 = variant(UNIT_10, "u10-0.54", MU_FRACTION=0.54, **NO_TIMING)
unit11 = variant(UNIT_11, "u11", **NO_TIMING)  # the shipped file (0.54), timing off
unit11_058 = variant(UNIT_11, "u11-0.58", MU_FRACTION=0.58, **NO_TIMING)

## 2. U11.6: held-out cars, at 0.54

Cars 35 and 41: the lowest-grip held-out car (grip x0.924) and one of the four held-out cars that crash at 0.58.

In [ ]:
for k in (35, 41):
    result = race(unit11, k)
    print(f"held-out car {k} (race seed {k % 5}): clean {clean(result)}, {result.total_time:.2f} s, "
          f"{line_error(result):.2f} m from its line")
    assert clean(result)  # U11.6 (the premise: 24/24 held-out cars clean)

## 3. U11.5 and U11.3: the fallback on a crashing car, and the restart is a fresh race

At 0.58 on training car 20 (race seed 0), Unit 10 crashes at 99.10 s and keeps crashing (the hook of the lesson); the shipped file falls back after the crash and finishes.
Its twin is a fresh race of the same car and race seed in the fallback configuration (`STEERING = "pursuit"`, `MU_FRACTION = 0.58 * 0.86`).

In [ ]:
Kept, agents = kept(unit11_058)
saved = race(Kept, 20)
(agent,) = agents
twin = race(variant(UNIT_11, "u11-twin", STEERING="pursuit", MU_FRACTION=0.58 * 0.86, **NO_TIMING), 20)
crash = saved.respawn_times[0]
remainder = saved.total_time - crash - HOLD_S
print(f"car 20, f = 0.58: finished {saved.finished}, {saved.total_time:.2f} s, {saved.collisions} collision at "
      f"{crash:.2f} s; respawns handled {agent.respawns}, mode {agent.mode!r}, limit_scale {agent.limit_scale}")
print(f"the race after the restart {remainder:.2f} s; the fresh twin {twin.total_time:.2f} s "
      f"(difference {remainder - twin.total_time:+.2f} s)")
assert saved.finished and saved.collisions == 1 and abs(saved.total_time - 385.05) <= TICKS_2  # U11.5
assert abs(crash - 99.10) <= TICKS_2
assert agent.respawns == saved.collisions == 1 and agent.mode == "fallback" and agent.limit_scale == 0.86
assert twin.finished and twin.collisions == 0 and abs(twin.total_time - 285.35) <= TICKS_2
assert abs(remainder - twin.total_time) <= 1.0 and remainder < twin.total_time  # U11.3, and its sign

## 4. U11.5: the lag-2 car

`carla_like(noise=True, lag_ticks=2)`: the unperturbed car with one tick more actuator lag than Unit 8 identified.
Unit 10's MPC models a lag of 1 tick and crash-loops at every fraction the probes tried; the fallback's pure pursuit has no lag model.

In [ ]:
loop = race(unit10_054, lag_ticks=2)
Kept, agents = kept(unit11)
survived = race(Kept, lag_ticks=2)
(agent,) = agents
print(f"lag 2, Unit 10 at 0.54: finished {loop.finished}, {loop.collisions} collisions in {loop.total_time:.0f} s")
print(f"lag 2, Unit 11 at 0.54: finished {survived.finished}, {survived.total_time:.2f} s, "
      f"{survived.collisions} collision; mode {agent.mode!r}")
assert not loop.finished and loop.collisions >= 3  # U11.5: a crash loop
assert survived.finished and survived.collisions == 1 and abs(survived.total_time - 321.65) <= TICKS_2  # U11.5
assert agent.respawns == 1 and agent.mode == "fallback"

## 5. U11.7: with no respawn, Unit 11 is Unit 10

The unperturbed car, race seed 0, both files at 0.54: every telemetry row must be equal.

In [ ]:
Kept, agents = kept(unit11)
eleven = race(Kept)
(agent,) = agents
ten = race(unit10_054)
same = all(np.sum(eleven.telemetry[name] != ten.telemetry[name]) == 0 for name in eleven.telemetry.dtype.names)
print(f"unperturbed car, f = 0.54: Unit 11 {eleven.total_time:.2f} s, Unit 10 {ten.total_time:.2f} s; "
      f"identical telemetry: {same}; respawns {agent.respawns}, warnings {agent.warnings}, mode {agent.mode!r}")
assert clean(eleven) and abs(eleven.total_time - 279.65) <= TICKS_2  # U11.7
assert same and eleven.total_time == ten.total_time and eleven.collisions == ten.collisions == 0
assert agent.respawns == 0 and agent.warnings == 0 and agent.mode == "primary" and agent.limit_scale == 1.0

## 6. U11.7a: the handler, race-free

A fake collision sensor feeds the shipped `respawned()` one reading per step, through a subclass that calls the shipped methods the way `step()` does (the step counter `ticks` set by hand, as `step()` sets it; a handled respawn records its step, as `step()` does).
racekit reports an impulse on one step; CARLA may repeat it.
Then `fall_back()` on a fresh agent: the MPC's memory is cleared, the re-anchor is set for the next step, the steering mode is the fallback's, and every section's lateral limit and the speed profile come down.

In [ ]:
class FakeSensor:
    def __init__(self, value):
        self.value = np.array(value, dtype=float)

    def get_last_gym_observation(self):
        return self.value


class Scripted(Unit11):
    # The shipped class, driven by hand: its own methods, called the way step() calls them.
    def feed(self, sensor, script):
        # One reading per step; returns the steps on which a respawn was handled.
        handled = []
        for tick, reading in enumerate(script, start=1):
            sensor.value = np.array(reading, dtype=float)
            self.ticks = tick
            if self.respawned():
                self.respawn_tick = tick
                handled.append(tick)
        return handled

    def limits(self):
        return [self.lateral_limit(section) for section in range(6)]

    def degrade(self):
        self.fall_back()


QUIET, CRASH = (0.0, 0.0, 0.0), (0.0, -1000.0, 0.0)
sensor = FakeSensor(QUIET)
agent = Scripted(track.waypoints, None, collision_sensor=sensor)
script = [QUIET] + [CRASH] * 10 + [QUIET] * 9 + [CRASH] + [QUIET] * 20 + [CRASH]  # steps 1-42
handled = agent.feed(sensor, script)
print("steps handled:", handled)
assert handled == [2, 42]  # the repeat (steps 3-11) and the impulse inside the window (step 21) are not handled

agent = Scripted(track.waypoints, None)
limits = agent.limits()
profile = np.array(agent.profile)
agent.mpc_queue, agent.mpc_last = [0.02], 0.02
agent.degrade()
print(f"after fall_back: mode {agent.mode!r}, reanchor {agent.reanchor}, queue {agent.mpc_queue}, "
      f"limit_scale {agent.limit_scale}; the fastest profile speed {np.max(profile):.2f} -> {np.max(agent.profile):.2f} m/s, "
      f"the slowest {np.min(profile):.2f} -> {np.min(agent.profile):.2f} m/s")
assert agent.mode == "fallback" and agent.reanchor and agent.mpc_queue == [0.0] and agent.mpc_last == 0.0
assert agent.plan is None and agent.limit_scale == 0.86
assert np.allclose(agent.limits(), 0.86 * np.array(limits), rtol=1e-12)
assert np.max(np.array(agent.profile) - profile) <= 1e-12 and np.min(agent.profile) < np.min(profile)

report = check_submission(UNIT_11)  # the shipped file, as shipped (MPC_BUDGET_MS = 5.0)
print(report)
assert report.ok, report.findings

## Unit 11, pinned

| `test-circuit`, `carla_like(noise=True, ...)` | Unit 10 | Unit 11 (`agents/unit-11.py`) |
|---|---|---|
| held-out cars 35 and 41, at 0.54 | — | clean |
| training car 20, at 0.58 | a crash loop (lesson) | 385.05 s, 1 collision (restart within 1 s of a fresh race) |
| the lag-2 car, at 0.54 | does not finish | 321.65 s, 1 collision |
| the unperturbed car, at 0.54 | 279.65 s | 279.65 s, row for row |

The routed premise races the full families: Unit 11 is clean on 24/24 held-out perturbations at 0.54, finishes every one of the 48 perturbed races at 0.58 with at most one collision, and its warnings, used as a trigger, do not prevent a single crash.